In [1]:
import sys
from pathlib import Path

root = Path.cwd().parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from src.data_loader import load_data 
from src.compare_models import (
    compare_models,
    save_results,
    plot_results,
    save_best_model,
)
from src.feature_engineering import (
    split_data,
    get_feature_types,
    create_preprocessor,
    transform_data,
)
from src.tuning import *

In [2]:
df = load_data(
    "../data/processed/clean_data.csv"
)

In [3]:
TARGET = "Churn Label"
df[TARGET] = df[TARGET].map({
    "No":0,
    "Yes":1
})
X = df.drop(columns=[TARGET])
y = df[TARGET]

In [4]:
X_train, X_test, y_train, y_test = split_data(X,y)
num_cols, cat_cols = get_feature_types(X_train)
preprocessor = create_preprocessor(
    num_cols,
    cat_cols,
)
X_train_processed, X_test_processed = transform_data(
    preprocessor,
    X_train,
    X_test,
)

In [5]:
rf = RandomForestClassifier(
    random_state=42,
)

cv = cross_validate_model(
    rf,
    X_train_processed,
    y_train,
)

cv

{'mean': np.float64(0.847155519752455),
 'std': np.float64(0.012649382362099218),
 'scores': [0.8591924773399253,
  0.8563771347325223,
  0.8565104292892571,
  0.8344946116685248,
  0.8292029457320452]}

In [6]:
grid = tune_random_forest(
    X_train_processed,
    y_train,
)

In [7]:
grid.best_score_

np.float64(0.8562877393174799)

In [8]:
grid.best_params_

{'max_depth': 20,
 'min_samples_leaf': 1,
 'min_samples_split': 5,
 'n_estimators': 200}

In [9]:
save_best_model(
    grid,
    "../models/best_model.pkl",
)

In [10]:
save_best_params(
    grid,
    "../reports/best_params.json",
)

In [11]:
save_cv_results(
    grid,
    "../reports/tuning_results.csv",
)

In [12]:
plot_top_results(
    grid,
    "../reports/tuning_results.png",
)